# AK-Byte-1 — Google Colab GPU Training
This notebook trains the repository's own randomly initialized Transformer from scratch. It does not use a hosted LLM API or pretrained weights. Profile: 12 layers, 512 hidden dimensions, 8 attention heads. The starter dataset is NOT one million examples; only real, reviewed, properly licensed data may be added. A free GPU/session is not guaranteed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, torch
REPO_URL = 'https://github.com/sidharthkumar8580-oss/Ankit-Kryptex-.git'
if not os.path.exists('/content/Ankit-Kryptex-'):
    subprocess.run(['git','clone',REPO_URL,'/content/Ankit-Kryptex-'], check=True)
os.chdir('/content/Ankit-Kryptex-')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda': print(torch.cuda.get_device_name(0))
else: print('No GPU assigned. In Colab choose Runtime > Change runtime type > T4 GPU (if available).')

In [ ]:
from model.config import BLOCK_SIZE
from model.network import AKByteModel
import json, random, time
random.seed(42); torch.manual_seed(42)
if DEVICE.type == 'cuda': torch.cuda.manual_seed_all(42)
examples=[]
with open('data/train.jsonl',encoding='utf-8') as f:
    for line in f:
        if not line.strip(): continue
        item=json.loads(line)
        prompt=('User: '+item['prompt'].strip()+'\nAnkit Kryptex: ').encode('utf-8')
        answer=(item['response'].strip()+'\n\n').encode('utf-8')
        answer=answer[:max(1,BLOCK_SIZE-len(prompt))]
        seq=torch.tensor(list(prompt+answer),dtype=torch.long)
        plen=min(len(prompt),len(seq)-1)
        if len(seq)>=3 and plen<len(seq)-1: examples.append((seq,plen))
print('Training examples actually loaded:',len(examples),'| context bytes:',BLOCK_SIZE)\nprint('Target dataset size: 1,000,000 genuine examples; target is NOT a claim about current data.')
assert examples, 'Dataset mein usable examples nahi mile.'
model=AKByteModel().to(DEVICE)
print('Parameters:',sum(p.numel() for p in model.parameters()))

## Training settings
Use a cautious T4-friendly batch size. The model has been enlarged, but a million-example dataset is not included: add genuine, deduplicated, licensed examples before long training. Increasing steps on a tiny dataset only repeats examples and can overfit.

In [ ]:
STEPS = 1000
BATCH_SIZE = 2
LEARNING_RATE = 1e-4
SAVE_DIR='/content/drive/MyDrive/AK-Byte-1'
os.makedirs(SAVE_DIR,exist_ok=True)
LATEST=os.path.join(SAVE_DIR,'checkpoint_latest.pt')
FINAL=os.path.join(SAVE_DIR,'weights.pt')
optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=0.01)
start_step=0
if os.path.exists(LATEST):
    choice=input('Existing checkpoint found. Resume from it? type yes/no: ').strip().lower()
    if choice=='yes':
        ckpt=torch.load(LATEST,map_location=DEVICE,weights_only=False)
        model.load_state_dict(ckpt['state_dict']); optimizer.load_state_dict(ckpt['optimizer'])
        start_step=int(ckpt.get('step',0)); print('Resuming at step',start_step)
model.train(); running=0.0; started=time.time()
for step in range(start_step+1,start_step+STEPS+1):
    chosen=random.choices(examples,k=BATCH_SIZE)
    x=torch.zeros((BATCH_SIZE,BLOCK_SIZE),dtype=torch.long)
    y=torch.full((BATCH_SIZE,BLOCK_SIZE),-100,dtype=torch.long)
    for row,(seq,plen) in enumerate(chosen):
        inp=seq[:-1]; tgt=seq[1:]; n=min(len(inp),BLOCK_SIZE)
        x[row,:n]=inp[:n]
        first=max(0,plen-1); end=min(len(tgt),BLOCK_SIZE)
        if first<end: y[row,first:end]=tgt[first:end]
    x=x.to(DEVICE); y=y.to(DEVICE)
    _,loss=model(x,y)
    optimizer.zero_grad(set_to_none=True); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
    running+=loss.item()
    if step%100==0 or step==start_step+STEPS:
        print(f'Step {step} | loss {running/min(100,step-start_step):.4f} | elapsed {(time.time()-started)/60:.1f} min')
        running=0.0
    if step%250==0:
        torch.save({'state_dict':model.state_dict(),'optimizer':optimizer.state_dict(),'step':step,'examples':len(examples),'format':'ak-byte-1-colab'},LATEST)
torch.save({'state_dict':model.state_dict(),'steps':start_step+STEPS,'examples':len(examples),'context_length':BLOCK_SIZE,'format':'ak-byte-1-sft-v2'},FINAL)
torch.save({'state_dict':model.state_dict(),'optimizer':optimizer.state_dict(),'step':start_step+STEPS,'examples':len(examples),'format':'ak-byte-1-colab'},LATEST)
print('Final weights saved to:',FINAL)

In [ ]:
# Quick local test of the trained checkpoint
model.eval()
question='Namaste, tum kaun ho?'
prefix=('User: '+question+'\nAnkit Kryptex: ').encode('utf-8')
tokens=torch.tensor(list(prefix),dtype=torch.long,device=DEVICE)[None,:]
with torch.no_grad(): out=model.generate(tokens,max_new_tokens=80)
print(bytes(out[0,tokens.shape[1]:].tolist()).decode('utf-8',errors='replace'))
print('\nWeights are in Google Drive > MyDrive > AK-Byte-1 > weights.pt')
print('To use them in the website/API, the weights file must be added to the repository at model/weights.pt, then redeploy.')

## Important
Keep the notebook tab open while training. Colab sessions can disconnect and free GPU availability/quotas vary. Periodic checkpoints are saved to Google Drive. The current training data is only a small starter set; more high-quality, varied, properly licensed examples are needed for meaningful improvement. Do not add private messages, passwords, API keys, or personal data to the dataset.